# Verification of `LinearSmallStrainFEMForceField`

This notebook verifies SOFA's `LinearSmallStrainFEMForceField` with the Method of Manufactured Solutions, presented in [IntroductionMMS.ipynb](IntroductionMMS.ipynb). It presents the governing PDEs and some of the cases used for verification, in `verification/cases/LinearSmallStrainFEMForceField/`.

## Governing PDE for Linear Elastic Materials

Under small-strain deformations, the displacement $\mathbf{u}$ of a linear elastic body satisfies

$$\mathcal{L}(\mathbf{u}) = -\nabla \cdot \boldsymbol{\sigma}(\mathbf{u}) = 0$$

This is the equation solved by the `LinearSmallStrainFEMForceField` component in SOFA. The body occupies a domain $\Omega$ in $d = 1$, $2$ or $3$ dimensions, and $\Gamma$ is its boundary.

The material is defined by its strain energy density $\psi$, the elastic energy stored per unit volume, and the stress is its derivative with respect to the displacement gradient. For a linear elastic material, $\psi$ depends on the strain $\boldsymbol{\varepsilon}$:

$$\boldsymbol{\varepsilon}(\mathbf{u}) = \frac{1}{2}\left(\nabla \mathbf{u} + \nabla \mathbf{u}^T\right), \qquad
  \psi(\boldsymbol{\varepsilon}) = \frac{\lambda}{2}\,\mathrm{tr}(\boldsymbol{\varepsilon})^2 + \mu\,\boldsymbol{\varepsilon} : \boldsymbol{\varepsilon}$$

$$\boldsymbol{\sigma}(\mathbf{u}) = \frac{\partial \psi}{\partial \nabla \mathbf{u}} = \lambda\,\mathrm{tr}(\boldsymbol{\varepsilon})\,\mathbf{I} + 2\mu\,\boldsymbol{\varepsilon}$$

The Lamé parameters $\lambda$ and $\mu$ are computed from Young's modulus $E$ and the Poisson ratio $\nu$:

$$\mu = \frac{E}{2(1 + \nu)}, \qquad \lambda = \frac{E\nu}{(1 + \nu)\,(1 - (d - 1)\,\nu)}$$

### Manufactured Solution and Boundary Conditions

For a manufactured solution $\hat{\mathbf{u}}$, with stress $\hat{\boldsymbol{\sigma}} = \boldsymbol{\sigma}(\hat{\mathbf{u}})$, the source term is

$$\mathbf{s} = \mathcal{L}(\hat{\mathbf{u}}) = -\nabla \cdot \hat{\boldsymbol{\sigma}}$$

The boundary conditions are taken from $\hat{\mathbf{u}}$ as well. The weak form shows which quantities a boundary condition can prescribe [[1]](#ref1). Multiplying $\mathcal{L}(\mathbf{u}) = \mathbf{s}$ by a test function $\mathbf{v}$, integrating over $\Omega$ and applying the divergence theorem gives

$$\int_\Omega \boldsymbol{\sigma} : \boldsymbol{\varepsilon}(\mathbf{v})\,d\Omega - \int_\Gamma \mathbf{t} \cdot \mathbf{v}\,d\Gamma
  = \int_\Omega \mathbf{s} \cdot \mathbf{v}\,d\Omega$$

where $\mathbf{t} = \boldsymbol{\sigma} \cdot \mathbf{n}$ is the traction and $\mathbf{n}$ the outward normal of $\Gamma$.

At each point of $\Gamma$ and in each direction, a boundary condition prescribes one of two quantities:

- the displacement, $\mathbf{u} = \hat{\mathbf{u}}$ (Dirichlet). The test function vanishes there, and so does the boundary term;
- the traction, $\mathbf{t} = \hat{\mathbf{t}} = \hat{\boldsymbol{\sigma}} \cdot \mathbf{n}$ (Neumann), which enters the boundary term as a known load.

Since $\hat{\mathbf{u}}$ is known everywhere, both $\hat{\mathbf{u}}$ and $\hat{\mathbf{t}}$ are known on the whole of $\Gamma$. Which one goes where is a choice, and a region of $\Gamma$ can have its displacement prescribed in one direction and its traction in another; enough of the displacement must be prescribed to keep the body from translating and rotating freely.

The modified problem is then

$$-\nabla \cdot \boldsymbol{\sigma}(\mathbf{u}) = \mathbf{s} \quad \text{in } \Omega, \qquad
  \mathbf{u} = \hat{\mathbf{u}} \quad \text{on } \Gamma_D, \qquad
  \boldsymbol{\sigma}(\mathbf{u}) \cdot \mathbf{n} = \hat{\mathbf{t}} \quad \text{on } \Gamma_N$$

with $\Gamma_D$ and $\Gamma_N$ the parts of the boundary where the displacement and the traction are prescribed, and $\hat{\mathbf{u}}$ is its exact solution. Each case below picks $\hat{\mathbf{u}}$, $\Omega$, $\Gamma_D$ and $\Gamma_N$.

## Running the Cases

The cell below loads the suite's runner and prepares the figures. Run it once before any case: every case cell below then runs on its own, in any order.

In [ ]:
# Make the plugin's packages importable from verification/docs
import sys
sys.path.insert(0, "../..")

# Hide the runner's printed report; figures still show
import os
sys.stdout = open(os.devnull, "w")

# Load the suite's runner
from verification.run import run

# For showing figures inline
%matplotlib inline

# For drawing several convergence figures side by side
import matplotlib.pyplot as plt

## Edge Elements on a 1D Bar

The bar starts at the origin and has length $L$ along $x$. The displacement $u$ is a scalar.

### Quadratic Function

$$\hat{u} = A\,x^2, \qquad \hat{\sigma} = 2EAx, \qquad s = -2EA$$

- `left` ($x = 0$): $\hat{u} = 0$
- `right` ($x = L$): $\hat{t} = 2EAL$

In [ ]:
# Case: 1D bar, quadratic manufactured solution
study = run("../cases/LinearSmallStrainFEMForceField/Quadratic1D.json")
# u_h along the bar at each level, coarse to fine
study.plot_fields()
study.plots["convergence"]

### Trigonometric Function

$$\hat{u} = A\sin(kx), \qquad \hat{\sigma} = EAk\cos(kx), \qquad s = EAk^2\sin(kx)$$

with $k = \frac{2\pi}{L}$.

- `left` ($x = 0$): $\hat{u} = 0$
- `right` ($x = L$): $\hat{t} = EAk$

In [ ]:
# Case: 1D bar, trigonometric manufactured solution
study = run("../cases/LinearSmallStrainFEMForceField/Trigonometric1D.json")
# u_h along the bar at each level, coarse to fine
study.plot_fields()
study.plots["convergence"]

## Triangle and Quadrilateral Elements on a 2D Beam

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$.

### Quadratic Function

$$
\hat{\mathbf{u}} = A\begin{pmatrix} x^2 + xy \\ y^2 + xy \end{pmatrix}, 
\qquad \mathbf{s} = -A \begin{pmatrix} 3\lambda + 5\mu \\ 3\lambda + 5\mu \end{pmatrix}, 
\qquad \hat{\boldsymbol{\sigma}} = A\begin{pmatrix} 3\lambda(x + y) + 2\mu(2x + y) & \mu(x + y) \\ \mu(x + y) & 3\lambda(x + y) + 2\mu(x + 2y) \end{pmatrix}
$$

- `left` ($x = 0$): $\hat{u}_x = 0$, $\hat{t}_y = -\mu A y$
- `bottom` ($y = 0$): $\hat{u}_y = 0$, $\hat{t}_x = -\mu A x$
- `right` ($x = L$): $\hat{\mathbf{t}} = A\,\big(3\lambda(L + y) + 2\mu(2L + y),\ \mu(L + y)\big)$
- `top` ($y = W$): $\hat{\mathbf{t}} = A\,\big(\mu(x + W),\ 3\lambda(x + W) + 2\mu(x + 2W)\big)$

In [ ]:
# Case: 2D rectangle, quadratic manufactured solution, triangles (left) and quadrilaterals (right)
tri = run("../cases/LinearSmallStrainFEMForceField/Quadratic2D_Tri.json")
quad = run("../cases/LinearSmallStrainFEMForceField/Quadratic2D_Quad.json")
figure, (triangles, quadrilaterals) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tri.plot(triangles)
quad.plot(quadrilaterals)

# Deformed shape at each level, coarse to fine
tri.plot_fields()
quad.plot_fields()

### Trigonometric Function

$$\hat{\mathbf{u}} = A\begin{pmatrix} \sin(kx)\cos(ky) \\ \cos(kx)\sin(ky) \end{pmatrix}, \qquad \mathbf{s} = 2k^2(\lambda + 2\mu)\,\hat{\mathbf{u}}, \qquad \hat{\boldsymbol{\sigma}} = 2Ak\begin{pmatrix} (\lambda + \mu)\cos(kx)\cos(ky) & -\mu\sin(kx)\sin(ky) \\ -\mu\sin(kx)\sin(ky) & (\lambda + \mu)\cos(kx)\cos(ky) \end{pmatrix}$$

with $k = \frac{2\pi}{L}$, the same along $x$ and $y$ since the cases use a square, $L = W$.

- `left` ($x = 0$): $\hat{\mathbf{u}} = A\,\big(0,\ \sin(ky)\big)$
- `bottom` ($y = 0$): $\hat{\mathbf{u}} = A\,\big(\sin(kx),\ 0\big)$
- `right` ($x = L$): $\hat{\mathbf{t}} = 2Ak(\lambda + \mu)\,\big(\cos(ky),\ 0\big)$
- `top` ($y = L$): $\hat{\mathbf{t}} = 2Ak(\lambda + \mu)\,\big(0,\ \cos(kx)\big)$

In [ ]:
# Case: 2D rectangle, trigonometric manufactured solution, triangles (left) and quadrilaterals (right)
tri = run("../cases/LinearSmallStrainFEMForceField/Trigonometric2D_Tri.json")
quad = run("../cases/LinearSmallStrainFEMForceField/Trigonometric2D_Quad.json")
figure, (triangles, quadrilaterals) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tri.plot(triangles)
quad.plot(quadrilaterals)

# Deformed shape at each level, coarse to fine
tri.plot_fields()
quad.plot_fields()

#### Plane Strain

The beam is solved in 3D space with $u_z = 0$. $\lambda$ and $\mu$ take their 3D values. The displacement is prescribed on all four sides:

- `left` ($x = 0$) and `right` ($x = L$): $\hat{\mathbf{u}} = A\,\big(0,\ \sin(ky),\ 0\big)$
- `bottom` ($y = 0$) and `top` ($y = L$): $\hat{\mathbf{u}} = A\,\big(\sin(kx),\ 0,\ 0\big)$

In [ ]:
# Case: 2D rectangle in plane strain, trigonometric manufactured solution, triangles
study = run("../cases/LinearSmallStrainFEMForceField/Trigonometric2D_Tri_PlaneStrain.json")
# Deformed shape at each level, coarse to fine
study.plot_fields()
study.plots["convergence"]

## Triangle and Quadrilateral Elements on a 2D Tire

The tire is an annulus centred at the origin, with inner radius $R_i$ and outer radius $R_o$, adjusted to 2D from [[2]](#ref2).

### Quartic Function

$$\hat{\mathbf{u}} = A\begin{pmatrix} x^4 + \frac{2}{5}xy \\ \frac{1}{10}y^4 - \frac{2}{5}xy \end{pmatrix}, \qquad \mathbf{s} = -\frac{2A}{5}\begin{pmatrix} 30(\lambda + 2\mu)x^2 - (\lambda + \mu) \\ 3(\lambda + 2\mu)y^2 + (\lambda + \mu) \end{pmatrix}, \qquad \hat{\boldsymbol{\sigma}} = A\begin{pmatrix} \lambda\theta + 2\mu\left(4x^3 + \frac{2}{5}y\right) & \frac{2}{5}\mu(x - y) \\ \frac{2}{5}\mu(x - y) & \lambda\theta + \frac{4}{5}\mu\left(y^3 - x\right) \end{pmatrix}$$

with $\theta = 4x^3 + \frac{2}{5}\left(y^3 + y - x\right)$.

The boundary is the inner and outer rims of the annulus:

- `inner` ($r = R_i$): Dirichlet, $\mathbf{u} = \hat{\mathbf{u}}$
- `outer` ($r = R_o$): Neumann, $\mathbf{t} = \hat{\boldsymbol{\sigma}} \cdot \mathbf{n}$

In [ ]:
# Case: 2D tire, quartic manufactured solution, triangles (left) and quadrilaterals (right)
tri = run("../cases/LinearSmallStrainFEMForceField/Quartic2D_Tri.json")
quad = run("../cases/LinearSmallStrainFEMForceField/Quartic2D_Quad.json")
figure, (triangles, quadrilaterals) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tri.plot(triangles)
quad.plot(quadrilaterals)

# Deformed shape at each level, coarse to fine
tri.plot_fields()
quad.plot_fields()

## Tetrahedron and Hexahedron Elements on a 3D Beam

The beam starts at the origin, with length $L$ along $x$, width $W$ along $y$ and height $H$ along $z$.

### Quadratic Function

$$\hat{\mathbf{u}} = A\begin{pmatrix} x^2 + xy + xz \\ y^2 + yz + xy \\ z^2 + zx + yz \end{pmatrix}, \qquad \mathbf{s} = -2A\begin{pmatrix} 2\lambda + 3\mu \\ 2\lambda + 3\mu \\ 2\lambda + 3\mu \end{pmatrix}, \qquad \hat{\boldsymbol{\sigma}} = A\begin{pmatrix} 4\lambda\theta + 2\mu(2x + y + z) & \mu(x + y) & \mu(x + z) \\ \mu(x + y) & 4\lambda\theta + 2\mu(x + 2y + z) & \mu(y + z) \\ \mu(x + z) & \mu(y + z) & 4\lambda\theta + 2\mu(x + y + 2z) \end{pmatrix}$$

with $\theta = x + y + z$.

- `left` ($x = 0$): $\hat{u}_x = 0$, $\hat{t}_y = -\mu A y$, $\hat{t}_z = -\mu A z$
- `bottom` ($y = 0$): $\hat{u}_y = 0$, $\hat{t}_x = -\mu A x$, $\hat{t}_z = -\mu A z$
- `front` ($z = 0$): $\hat{u}_z = 0$, $\hat{t}_x = -\mu A x$, $\hat{t}_y = -\mu A y$
- `right` ($x = L$): $\hat{\mathbf{t}} = A\,\big(4\lambda(L + y + z) + 2\mu(2L + y + z),\ \mu(L + y),\ \mu(L + z)\big)$
- `top` ($y = W$): $\hat{\mathbf{t}} = A\,\big(\mu(x + W),\ 4\lambda(x + W + z) + 2\mu(x + 2W + z),\ \mu(W + z)\big)$
- `back` ($z = H$): $\hat{\mathbf{t}} = A\,\big(\mu(x + H),\ \mu(y + H),\ 4\lambda(x + y + H) + 2\mu(x + y + 2H)\big)$

In [ ]:
# Case: 3D box, quadratic manufactured solution, tetrahedra (left) and hexahedra (right)
tet = run("../cases/LinearSmallStrainFEMForceField/Quadratic3D_Tetra.json")
hexa = run("../cases/LinearSmallStrainFEMForceField/Quadratic3D_Hexa.json")
figure, (tetrahedra, hexahedra) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tet.plot(tetrahedra)
hexa.plot(hexahedra)

# Deformed shape at each level, coarse to fine
tet.plot_fields()
hexa.plot_fields()

### Trigonometric Function

$$\hat{\mathbf{u}} = A\begin{pmatrix} \sin(kx)\cos(ky)\cos(kz) \\ \cos(kx)\sin(ky)\cos(kz) \\ \cos(kx)\cos(ky)\sin(kz) \end{pmatrix}, \qquad \mathbf{s} = 3k^2(\lambda + 2\mu)\,\hat{\mathbf{u}}, \qquad \hat{\boldsymbol{\sigma}} = Ak\begin{pmatrix} (3\lambda + 2\mu)\,\cos(kx)\cos(ky)\cos(kz) & -2\mu\,\sin(kx)\sin(ky)\cos(kz) & -2\mu\,\sin(kx)\cos(ky)\sin(kz) \\ -2\mu\,\sin(kx)\sin(ky)\cos(kz) & (3\lambda + 2\mu)\,\cos(kx)\cos(ky)\cos(kz) & -2\mu\,\cos(kx)\sin(ky)\sin(kz) \\ -2\mu\,\sin(kx)\cos(ky)\sin(kz) & -2\mu\,\cos(kx)\sin(ky)\sin(kz) & (3\lambda + 2\mu)\,\cos(kx)\cos(ky)\cos(kz) \end{pmatrix}$$

with $k = \frac{2\pi}{L}$, the same along $x$, $y$ and $z$ since the cases use a cube, $L = W = H$.

- `left` ($x = 0$): $\hat{\mathbf{u}} = A\,\big(0,\ \sin(ky)\cos(kz),\ \cos(ky)\sin(kz)\big)$
- `bottom` ($y = 0$): $\hat{\mathbf{u}} = A\,\big(\sin(kx)\cos(kz),\ 0,\ \cos(kx)\sin(kz)\big)$
- `front` ($z = 0$): $\hat{\mathbf{u}} = A\,\big(\sin(kx)\cos(ky),\ \cos(kx)\sin(ky),\ 0\big)$
- `right` ($x = L$): $\hat{\mathbf{t}} = Ak(3\lambda + 2\mu)\,\big(\cos(ky)\cos(kz),\ 0,\ 0\big)$
- `top` ($y = L$): $\hat{\mathbf{t}} = Ak(3\lambda + 2\mu)\,\big(0,\ \cos(kx)\cos(kz),\ 0\big)$
- `back` ($z = L$): $\hat{\mathbf{t}} = Ak(3\lambda + 2\mu)\,\big(0,\ 0,\ \cos(kx)\cos(ky)\big)$

In [ ]:
# Case: 3D box, trigonometric manufactured solution, tetrahedra (left) and hexahedra (right)
tet = run("../cases/LinearSmallStrainFEMForceField/Trigonometric3D_Tetra.json")
hexa = run("../cases/LinearSmallStrainFEMForceField/Trigonometric3D_Hexa.json")
figure, (tetrahedra, hexahedra) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tet.plot(tetrahedra)
hexa.plot(hexahedra)

# Deformed shape at each level, coarse to fine
tet.plot_fields()
hexa.plot_fields()

## Tetrahedron and Hexahedron Elements on a 3D Tire

The tire is the annulus of the 2D tire, extruded along $z$ from $0$ to $H$.

### Quartic Function

The manufactured solution is the one of [[2]](#ref2):

$$\hat{\mathbf{u}} = A\begin{pmatrix} x^4 - \frac{2}{5}yz \\ -\frac{1}{10}y^4 - \frac{2}{5}xyz \\ z^4 - \frac{2}{5}xy \end{pmatrix}, \qquad \mathbf{s} = A\begin{pmatrix} -12(\lambda + 2\mu)x^2 + \frac{2}{5}(\lambda + \mu)z \\ \frac{6}{5}(\lambda + 2\mu)y^2 \\ -12(\lambda + 2\mu)z^2 + \frac{2}{5}(\lambda + \mu)x \end{pmatrix}, \qquad \hat{\boldsymbol{\sigma}} = A\begin{pmatrix} \lambda\theta + 8\mu x^3 & -\frac{2}{5}\mu z(1 + y) & -\frac{4}{5}\mu y \\ -\frac{2}{5}\mu z(1 + y) & \lambda\theta - \frac{4}{5}\mu\left(y^3 + xz\right) & -\frac{2}{5}\mu x(1 + y) \\ -\frac{4}{5}\mu y & -\frac{2}{5}\mu x(1 + y) & \lambda\theta + 8\mu z^3 \end{pmatrix}$$

with $\theta = 4x^3 + 4z^3 - \frac{2}{5}y^3 - \frac{2}{5}xz$.

The boundary is the inner and outer curved surfaces and the two flat faces:

- `inner` ($r = R_i$): Dirichlet, $\mathbf{u} = \hat{\mathbf{u}}$
- `outer` ($r = R_o$): Dirichlet, $\mathbf{u} = \hat{\mathbf{u}}$
- `front` ($z = 0$): Neumann, $\mathbf{t} = \hat{\boldsymbol{\sigma}} \cdot \mathbf{n}$
- `back` ($z = H$): Neumann, $\mathbf{t} = \hat{\boldsymbol{\sigma}} \cdot \mathbf{n}$

In [ ]:
# Case: 3D tire, quartic manufactured solution, tetrahedra (left) and hexahedra (right)
tet = run("../cases/LinearSmallStrainFEMForceField/Quartic3D_Tetra.json")
hexa = run("../cases/LinearSmallStrainFEMForceField/Quartic3D_Hexa.json")
figure, (tetrahedra, hexahedra) = plt.subplots(1, 2, figsize=(14.4, 4.8))
tet.plot(tetrahedra)
hexa.plot(hexahedra)

# Deformed shape at each level, coarse to fine
tet.plot_fields()
hexa.plot_fields()

<a id="references"></a>
## References

<a id="ref1"></a>
**[1]** T. J. R. Hughes, *The Finite Element Method: Linear Static and Dynamic Finite Element Analysis*, Dover, Mineola, NY, 2000. ISBN 978-0-486-41181-1. &mdash; *Chapter 2 (the weak formulation of linear elasticity).*

<a id="ref2"></a>
**[2]** É. Chamberland, A. Fortin and M. Fortin, "Comparison of the performance of some finite element discretizations for large deformation elasticity problems," *Computers and Structures*, 88(11–12):664–673, 2010.